In [ ]:
import sys
print(sys.executable)


---

## ✅ **Task: Sentiment classification using with the given specifications**
 
Provide a complete, working Keras example that includes:

- an Embedding layer  
- a simple RNN layer  
- one Dense output layer with **sigmoid** activation  

Use a small sample text dataset and produce a **sentiment classification** output (positive/negative).  
Include all steps:

1. preparing the input text  
2. tokenization and sequence preparation  
3. building the model  
4. training the model  
5. evaluating the model  
6. running a prediction on a new sentence  

uses simple data so We can understand each step.

---

Later in following Task, we will do these :

- a **PyTorch** version  
- a **GRU/LSTM** version  
- a **Word2Vec‑based embedding** version  
- a **diagram explaining the architecture**

Here are a few ways to format this sequence as a Mermaid flow diagram for your notes:

### 🔹 Option 1: Horizontal Flow (Compact & Clean)

```mermaid
graph LR
    T["Text<br><i>(Raw Input)</i>"] -->|Tokenizer| ID["Token IDs<br><i>(Integer List)</i>"]
    ID -->|Fixed Length| P["Padding<br><i>(Equal Sequences)</i>"]
    P -->|Lookup| E["Embedding Layer<br><i>(Dense Vectors)</i>"]
    E -->|Time Steps| R["SimpleRNN Layer<br><i>(Recurrent Math)</i>"]
    R -->|Activation| D(["Dense Layer<br><i>(Sigmoid Output)</i>"])

    style E fill:#2563eb,color:#fff,stroke:#60a5fa,stroke-width:2px
    style D fill:#10b981,color:#fff,stroke:#34d399,stroke-width:2px
```

---

### 🔹 Option 2: Vertical Detailed Flow (With Code Snippets)

```mermaid
graph TD
    T["Text Input<br><code>text = ['good movie']</code>"] -->|Tokenization| ID["Token IDs<br><code>ids = [4, 12]</code>"]
    ID -->|Equalizing Sequence Length| P["Padded Sequences<br><code>padded = [4, 12, 0, 0]</code>"]
    P -->|Vector Lookup| E["Embedding Layer<br><code>Embedding(vocab, dim)</code>"]
    E -->|Sequence Processing| R["SimpleRNN Layer<br><code>SimpleRNN(units)</code>"]
    R -->|Binary Classification| D(["Dense (Sigmoid)<br><code>Dense(1, activation='sigmoid')</code>"])

    style E fill:#2563eb,color:#fff,stroke:#60a5fa,stroke-width:2px
    style D fill:#10b981,color:#fff,stroke:#34d399,stroke-width:2px
```

---



In [ ]:
import os 
import random

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer

from tensorflow.keras.layers import Dense, Embedding, SimpleRNN
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.sequence import pad_sequences
# from tesnorflow.keras.datasets import imdb


```python
pip install python-dotenv
from dotenv import load_dotenv 
load_dotenv()  #reads .env in current directory, loads into os.environ
print(os.environ["SEED"])           # → "42"  (still a string!)
import os
```
- load_dotenv() reads the file and does the equivalent of calling os.environ["SEED"] = "42" for every line. 
- after which your earlier code (str(SEED), etc.) 
- works the same as before.


In [ ]:
# 1. REPRODUCIBILITY
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# ============================================================
# 2. SMALL SAMPLE TEXT DATASET
# ============================================================

texts = [
    "I love this movie",
    "This film was excellent",
    "The story was wonderful",
    "The acting was very good",
    "I really enjoyed the movie",
    "This was an amazing film",
    "The movie was interesting",
    "I liked the characters",
    "The ending was beautiful",
    "This film made me happy",
    "I hate this movie",
    "This film was terrible",
    "The story was boring",
    "The acting was very bad",
    "I did not enjoy the movie",
    "This was an awful film",
    "The movie was disappointing",
    "I disliked the characters",
    "The ending was horrible",
    "This film made me angry",
]

# Positive = 1
# Negative = 0

In [ ]:
labels = np.array([
    1, 1, 1, 1, 1,
    1, 1, 1, 1, 1,
    0, 0, 0, 0, 0,
    0, 0, 0, 0, 0,
], dtype=np.float32)

print(f"Texts length: {len(texts)}")
print(f"Labels length: {len(labels)}")


In [ ]:
# ============================================================
# 3. SPLIT INTO TRAINING AND TEST DATA
# ============================================================

train_texts = texts[:8] + texts[10:18]
train_labels = np.concatenate([
    labels[:8],
    labels[10:18],
])

test_texts = texts[8:10] + texts[18:20]
test_labels = np.concatenate([
    labels[8:10],
    labels[18:20],
])

print("Number of training examples:", len(train_texts))
print("Number of test examples:", len(test_texts))

In [ ]:
# ============================================================
# 4. TOKENIZATION
# ============================================================

# num_words limits the maximum vocabulary size.
# oov_token represents words that were not seen during training.
tokenizer = Tokenizer(
    num_words=1000, # 0,1 is resrved for padding(0) and OOV tokens(1), only 998 words will be kept in the vocabulary.
    oov_token="<OOV>"
)

# Learn the vocabulary only from the training text.
tokenizer.fit_on_texts(train_texts)

print("\nWord index:")
print(tokenizer.word_index)     

 
# Convert sentences into sequences of integer token IDs.
train_sequences = tokenizer.texts_to_sequences(train_texts)
test_sequences = tokenizer.texts_to_sequences(test_texts)

print("\nExample sentence:")
print(train_texts[0])

print("\nToken sequence:")
print(train_sequences[0])

# `num_words` in Keras `Tokenizer` — Summary Notes

## What it is
`num_words` limits the tokenizer's vocabulary to the **top N most frequent words**
seen during `fit_on_texts()`. Words outside this limit are replaced by the
`oov_token` (Out-Of-Vocabulary) when converting text to sequences.

```python
tokenizer = Tokenizer(
    num_words=1000,
    oov_token="<OOV>"
)
```

- ID `0` → reserved for padding
- ID `1` → reserved for `<OOV>` (unknown words)
- IDs `2 ... num_words-1` → the most frequent real words

## Why it matters
| Without `num_words` limit | With `num_words` limit |
|---|---|
| Every unique word gets an ID (could be 50,000+) | Only common, informative words get IDs |
| Huge Embedding layer → more params, slower, overfits | Small Embedding layer → efficient, generalizes better |
| Rare words (typos, names) add noise | Rare words collapse into `<OOV>`, ignored as noise |

## Flow: how a word gets its ID

```mermaid
flowchart TD
    A["fit_on_texts(training_text)"] --> B["Count frequency of every word"]
    B --> C["Rank words: most frequent -> least frequent"]
    C --> D{"Is word rank < num_words?"}
    D -- Yes --> E["Assign it a real integer ID (2, 3, 4, ...)"]
    D -- No --> F["Word falls outside vocabulary"]
    F --> G["Mapped to oov_token ID = 1 during texts_to_sequences()"]
    E --> H["Stored in tokenizer.word_index"]
    F --> H
```

## Important subtlety
`tokenizer.word_index` stores **ALL** words seen, regardless of `num_words`.
The cutoff is only enforced later, in `texts_to_sequences()`.

```mermaid
flowchart LR
    subgraph Build["fit_on_texts()"]
        direction LR
        W1["word_index dict"] --- N1["contains EVERY word, no cutoff"]
    end
    subgraph Convert["texts_to_sequences()"]
        direction LR
        W2["applies num_words cutoff HERE"] --- N2["rare words -> OOV id"]
    end
    Build --> Convert
```

## Applied to this notebook
```python
vocab_size = min(1000, len(tokenizer.word_index) + 1)
```

```mermaid
flowchart LR
    A["num_words = 1000 (generous cap)"] --> C{"min(1000, actual_vocab+1)"}
    B["actual training vocab = 31 words -> +1 = 32"] --> C
    C --> D["vocab_size = 32"]
    D --> E["Embedding layer only allocates 32 vectors"]
```

Since this dataset only has 31 unique words, `num_words=1000` never actually
kicks in — it's just a safe upper bound. The real limiting factor here is the
tiny dataset itself, not the `num_words` setting.

## One-line takeaway
> `num_words` is a **vocabulary-size control**: it keeps the model focused on
> common, meaningful words and treats rare/unseen words as `<OOV>`, keeping the
> Embedding layer small and the model less prone to overfitting.


```python
matrix = [
    [10, 20, 30],  # Row 0
    [40, 50, 60],  # Row 1
    [70, 80, 90]   # Row 2
]
print(len(matrix))
l_list = []
for i,row in enumerate(train_sequences):
    # print(f"Row {i}: {row}")
    curr_length = len(row)
    l_list.append(curr_length)
    print(f"Length of row {i}= {curr_length}", end=";")
print(f"\nLengths of max sequence: {max(l_list)}")
```

In [ ]:
print(f"\n train_sequence:{type(train_sequences)} \n value:{train_sequences[0]}")
print(f"full train_sequences:{train_sequences}")

In [ ]:
# ============================================================
# 5. PAD THE SEQUENCES
# ============================================================

MAX_SEQUENCE_LENGTH = 8

train_padded = pad_sequences(
    train_sequences,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

test_padded = pad_sequences(
    test_sequences,
    maxlen=MAX_SEQUENCE_LENGTH,
    padding="post",
    truncating="post"
)

print("\nPadded training sequence:")
print(train_padded[0])

print("\nTraining input shape:")
print(train_padded.shape)

In [ ]:
# ============================================================
# 6. BUILD THE MODEL
# ============================================================

vocab_size = min(
    1000,
    len(tokenizer.word_index) + 1
)

embedding_dimension = 16
rnn_units = 16

model = Sequential([
    # Converts each integer token into a dense vector.
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dimension,
        mask_zero=True
    ),

    # Reads the sequence of embedding vectors.
    SimpleRNN(
        units=rnn_units
    ),

    # Produces one probability between 0 and 1.
    Dense(
        units=1,
        activation="sigmoid"
    )
    ])

# `embedding_dimension` and `rnn_units` — Summary Notes

## What they are
Both are **hyperparameters** — chosen by you, not learned by training — that
control the *size/capacity* of two layers.

```python
embedding_dimension = 16
rnn_units = 16
```

| Parameter | Controls | Shape it defines |
|---|---|---|
| `embedding_dimension` | numbers used to represent **each word** | `Embedding` lookup table = `(vocab_size, 16)` |
| `rnn_units` | size of the RNN's **memory / hidden state** | hidden state vector = `(16,)`, updated every timestep |

## Data flow: how they connect

```mermaid
flowchart LR
    A["Word ID (int)<br/>e.g. 13 = 'love'"] --> B["Embedding lookup table<br/>(vocab_size x 16)"]
    B --> C["Word vector<br/>size = embedding_dimension (16)"]
    C --> D["SimpleRNN<br/>processes one word at a time"]
    D --> E["Hidden state updated each step<br/>size = rnn_units (16)"]
    E --> F["Final hidden state<br/>(16 numbers = sentence summary)"]
    F --> G["Dense(1, sigmoid)"]
    G --> H["Probability (0-1)"]
```

## Why size matters (capacity vs. overfitting trade-off)

```mermaid
flowchart TD
    S{"Choose dimension size"}
    S -->|"Too small"| U["Underfitting<br/>not enough room to capture patterns"]
    S -->|"Too large"| O["Overfitting<br/>too many params for small data,<br/>memorizes instead of generalizes"]
    S -->|"Just right"| G["Good fit<br/>matches dataset size/complexity"]
```

## Why `16` was chosen here
Dataset = 20 sentences, 31 unique words → very small.

```mermaid
flowchart LR
    A["vocab_size = 32"] --> B["embedding params = 32 x 16 = 512"]
    C["train examples = 16 sentences"] -.->|"already comparable to param count"| B
    B --> D["Bigger dims (e.g. 256) would\nmassively overfit this tiny data"]
```

`16`/`16` is simply a **small, sensible starting point** matched to a toy-scale
dataset — not a value with special significance.

## Practical sizing guide

| Situation | Typical `embedding_dimension` / `rnn_units` |
|---|---|
| Toy/tiny dataset (this notebook) | 8 – 32 |
| Small real dataset (hundreds–thousands of examples) | 32 – 128 |
| Large real dataset (e.g. IMDB, 25k+ reviews) | 64 – 300 |

## One-line takeaway
> Both parameters trade off **model capacity** against **overfitting risk** —
> pick them relative to how much data and vocabulary complexity you actually
> have, then tune based on validation performance.


In [ ]:
# ============================================================
# 7. COMPILE THE MODEL
# ============================================================

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


# Build the model so the summary displays all shapes.
model.build(
    input_shape=(None, MAX_SEQUENCE_LENGTH)
)

print("\nModel summary:")
model.summary()

In [ ]:
# ============================================================
# 8. TRAIN THE MODEL
# ============================================================

history = model.fit(
    train_padded,
    train_labels,
    epochs=40,
    batch_size=4,
    validation_split=0.25,
    verbose=1
)


In [ ]:
# ============================================================
# 9. EVALUATE THE MODEL
# ============================================================

test_loss, test_accuracy = model.evaluate(
    test_padded,
    test_labels,
    verbose=0
)

print("\nTest loss:", round(test_loss, 4))
print("Test accuracy:", round(test_accuracy, 4))


In [ ]:
# ============================================================
# 10. PREDICTION FUNCTION
# ============================================================

def predict_sentiment(sentence: str) -> None:
    """
    Convert a sentence into a padded token sequence,
    run the model, and print the sentiment result.
    """

    sequence = tokenizer.texts_to_sequences([sentence])

    padded_sequence = pad_sequences(
        sequence,
        maxlen=MAX_SEQUENCE_LENGTH,
        padding="post",
        truncating="post"
    )

    probability = float(
        model.predict(
            padded_sequence,
            verbose=0
        )[0][0]
    )

    sentiment = (
        "Positive"
        if probability >= 0.5
        else "Negative"
    )

    print("\nNew sentence:", sentence)
    print("Token sequence:", sequence[0])
    print("Padded sequence:", padded_sequence[0])
    print("Positive probability:", round(probability, 4))
    print("Predicted sentiment:", sentiment)


In [ ]:
# ============================================================
# 11. RUN PREDICTIONS
# ============================================================

predict_sentiment(
    "I really love this wonderful movie"
)

predict_sentiment(
    "This movie was boring and terrible"
)

###
---
tf.keras.layers.TextVectorization — this genuinely is the modern, recommended replacement for the old Tokenizer + pad_sequences combo you used. 
- TensorFlow's own docs mark keras.preprocessing.text.Tokenizer as legacy now. Let me teach you that one.

- Why TextVectorization replaced Tokenizer ?
Old way (your notebook)	New way (TextVectorization)
Tokenizer (builds vocab) + pad_sequences (pads) = 2 separate steps, done outside the model	One single Keras layer that does both, and can live inside the model
Model only accepts integer arrays as input	Model can accept raw text strings directly
- You must save/reload the tokenizer object separately from the model	Vocabulary is saved as part of the model automatically
- OOV token set manually (oov_token="<OOV>")	OOV handling is automatic/built-in (always index 1)
Code — rewriting your tokenize + pad steps
```python
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization, Embedding, SimpleRNN, Dense
from tensorflow.keras.models import Sequential

MAX_SEQUENCE_LENGTH = 8
VOCAB_SIZE = 1000

# This ONE layer replaces both Tokenizer() and pad_sequences()
vectorize_layer = TextVectorization(
    max_tokens=VOCAB_SIZE,                   # same role as Tokenizer's num_words
    standardize="lower_and_strip_punctuation",# lowercases + removes punctuation automatically
    output_mode="int",                # output = integer IDs (like texts_to_sequences)
    output_sequence_length=MAX_SEQUENCE_LENGTH 
    # same role as pad_sequences(maxlen=...),max_len = 4  # Sequence length to pad the outputs to 4.
)
```
###  References link for deatils of: [TextVectorization](https://www.tensorflow.org/api_docs/python/tf/keras/layers/TextVectorization)

```python
# "adapt" = the new "fit_on_texts" — scans training text, builds vocabulary
vectorize_layer.adapt(train_texts)

print("Vocabulary sample:", vectorize_layer.get_vocabulary()[:10])
# ['', '[UNK]', 'the', 'was', ...]
#  ^index0   ^index1 (this is the OOV token, built-in, always here)

# Convert straight to padded integer sequences, in one call
train_padded = vectorize_layer(train_texts)
test_padded = vectorize_layer(test_texts)

print("Example sentence:", train_texts[0])
print("Vectorized + padded:", train_padded[0].numpy())
```
- How it works, step by step
TextVectorization(...) 
— creates an empty layer, just configured with settings (max vocab size, how to standardize text, output length). No vocabulary yet.
.adapt(train_texts) 
— this is the direct replacement for tokenizer.fit_on_texts(). 
- It reads all training sentences once, lowercases + strips punctuation, splits into words, counts frequency, and builds the internal vocabulary (word → integer ID table), keeping only the top max_tokens most frequent words.

- Calling vectorize_layer(train_texts) — replaces texts_to_sequences() and pad_sequences() in one shot: text → word IDs → padded/truncated to output_sequence_length.

Index 0 is always padding, index 1 is always the built-in unknown-word token ([UNK]) — you don't configure oov_token yourself anymore, it's automatic.
The real superpower: put it inside the model
```python
model = Sequential([
    vectorize_layer,                     # <-- accepts RAW STRINGS now
    Embedding(input_dim=VOCAB_SIZE, output_dim=16, mask_zero=True),
    SimpleRNN(units=16),
    Dense(units=1, activation="sigmoid")
])

model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

# No manual tokenizing/padding needed anymore — feed raw sentences directly!
model.fit(train_texts, train_labels, epochs=40, batch_size=4, validation_split=0.25)

# Prediction also takes raw text directly, no predict_sentiment() helper needed for preprocessing
model.predict(["I really love this wonderful movie"])
```
Because vectorize_layer is now literally a layer in the model, when you later model.save(), the vocabulary is saved inside the model file too — no separate tokenizer.pkl to manage, and no risk of a mismatched tokenizer when you reload the model elsewhere.

---
